Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10e1daf90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10e1dba10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10e1daf90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10e1dba10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description

In [5]:
response=model_with_structure.invoke("Provide details about the moview Chennai Express")
response

Movie(title='Chennai Express', year=2013, director='Rohit Shetty', rating=8.5)

In [6]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Koi mil gya")
response

MovieDetails(title='Koi Mil Gaya', year=2003, cast=[Actor(name='Hrithik Roshan', role='Raju'), Actor(name='Priyanka Chopra', role='Isha'), Actor(name='Boman Irani', role='Mr. Sinha')], genres=['Drama', 'Romance', 'Fantasy'], budget=5.0)

In [7]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers endgame")
response

{'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [8]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Jawan")
response

{'budget': 200000000,
 'cast': [{'name': 'Shah Rukh Khan', 'role': 'Jawan'},
  {'name': 'Nayanthara', 'role': 'Zulekha'},
  {'name': 'Vijay Sethupathi', 'role': 'Inspector Vijay'}],
 'genres': ['Action', 'Thriller', 'Drama'],
 'title': 'Jawan',
 'year': 2023}

DataClasses

In [20]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [21]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"]
)

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=llm
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Adhiraj Singh, adhirajsingh@gmail.com, 98929699"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: Adhiraj Singh, adhirajsingh@gmail.com, 98929699', additional_kwargs={}, response_metadata={}, id='52c1889a-3343-422b-8b0e-c51cf16612ae'),
  AIMessage(content='Here is the extracted contact information:\n\n* Email: adhirajsingh@gmail.com\n* Phone: 98929699\n* Name: Adhiraj Singh', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 57, 'total_tokens': 91, 'completion_time': 0.106218479, 'completion_tokens_details': None, 'prompt_time': 0.008457373, 'prompt_tokens_details': None, 'queue_time': 0.054960783, 'total_time': 0.114675852}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f0f92-9dbd-7120-8945-0c7285d8049a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 34, 'total_tokens': 91})]}

In [24]:
result["messages"][-1].content

'Here is the extracted contact information:\n\n* Email: adhirajsingh@gmail.com\n* Phone: 98929699\n* Name: Adhiraj Singh'

In [25]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"]
)

structured_llm = llm.with_structured_output(ContactInfo)

result = structured_llm.invoke(
    "Extract contact info from: Adhiraj Singh, adhirajsingh@gmail.com, 98929699"
)

print(result)

name='Adhiraj Singh' email='adhirajsingh@gmail.com' phone='98929699'


In [28]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"]
)

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


aagent = create_agent(
    model=llm,
    response_format=ContactInfo
    
)


result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["messages"][-1].content

'Here is the extracted contact information:\n\n* Email: john@example.com\n* Phone: (555) 123-4567'